In [3]:
#q1
import pandas as pd
roll="1024170068"
fixed_entries = [
    {   "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing"
    }
]
#billing and general
personalized_entries = [
    {
        "question": "how can i download my fee receipt",
        "answer": "Go to Payments > History and click download receipt.",
        "keywords": "receipt invoice download bill",
        "category": "billing"
    },
    {
        "question": "where is the main campus located",
        "answer": "The main campus is located at Block B, University Road.",
        "keywords": "location address campus physical",
        "category": "general"
    }
]
all_entries = fixed_entries + personalized_entries
df = pd.DataFrame(all_entries)
print(df)


                            question  \
0             what is the annual fee   
1              how to reset password   
2        what are your working hours   
3              how can i pay the fee   
4  how can i download my fee receipt   
5   where is the main campus located   

                                              answer  \
0                          The annual fee is Rs 500.   
1                   Go to Settings > Reset Password.   
2                          We are open 9 AM to 5 PM.   
3         You can pay via UPI, card, or net banking.   
4  Go to Payments > History and click download re...   
5  The main campus is located at Block B, Univers...   

                           keywords category  
0             fee cost price charge  billing  
1              password reset login  account  
2            hours timing open time  general  
3               pay payment upi fee  billing  
4     receipt invoice download bill  billing  
5  location address campus physical  general

In [6]:
#q2
def score_hypothesis(query, dataframe):
    query_words = set(query.lower().split())
    scored_results = []

    for idx, row in dataframe.iterrows():
        kw_set = set(row['keywords'].lower().split())
        score = len(query_words.intersection(kw_set))
        if score > 0:
            scored_results.append({
                "question": row['question'],
                "answer": row['answer'],
                "category": row['category'],
                "confidence_score": score
            })

    results_df = pd.DataFrame(scored_results)
    if not results_df.empty:
        results_df = results_df.sort_values(by="confidence_score", ascending=False).reset_index(drop=True)
    return results_df
query = "how can I pay my fee"
print(f"Query: '{query}'")
print(score_hypothesis(query, df))

Query: 'how can I pay my fee'
                 question                                      answer  \
0   how can i pay the fee  You can pay via UPI, card, or net banking.   
1  what is the annual fee                   The annual fee is Rs 500.   

  category  confidence_score  
0  billing                 2  
1  billing                 1  


In [7]:
#q3
def same_category(category_name, dataframe):
    return dataframe[dataframe['category'] == category_name][['question', 'answer', 'category']].reset_index(drop=True)
print(same_category("billing", df))


                            question  \
0             what is the annual fee   
1              how can i pay the fee   
2  how can i download my fee receipt   

                                              answer category  
0                          The annual fee is Rs 500.  billing  
1         You can pay via UPI, card, or net banking.  billing  
2  Go to Payments > History and click download re...  billing  


In [8]:
#q4
new_keyword = "statement"
df.at[4, 'keywords'] += f" {new_keyword}"

csv_filename = f"{roll}_faq_data.csv"
df.to_csv(csv_filename, index=False)
print(df.loc[4, ['question', 'keywords']])

question          how can i download my fee receipt
keywords    receipt invoice download bill statement
Name: 4, dtype: object


In [9]:
#q5
category_counts = df.groupby('category').size()
print(category_counts)

category
account    1
billing    3
general    2
dtype: int64


In [13]:
#q6
def score_hypothesis_v2(query, dataframe):
    query_words = set(query.lower().split())
    scored_results = []

    for idx, row in dataframe.iterrows():
        kw_set = set(row['keywords'].lower().split())
        score = len(query_words.intersection(kw_set))
        if score > 0:
            scored_results.append({
                "question": row['question'],
                "answer": row['answer'],
                "category": row['category'],
                "confidence_score": score
            })

    if not scored_results:
        print("No matches found.")
        return

    results_df = pd.DataFrame(scored_results)
    max_score = results_df['confidence_score'].max()
    top_matches = results_df[results_df['confidence_score'] == max_score].reset_index(drop=True)

    if len(top_matches) > 1:
        print(f"Tie detected! Found {len(top_matches)} entries tied for top score ({max_score}):")
    else:
        print(f"Single top match found with score {max_score}:")

    print(top_matches[['question', 'answer', 'confidence_score']])


score_hypothesis_v2("what is the fee", df)
score_hypothesis_v2("password reset", df)

Tie detected! Found 2 entries tied for top score (1):
                 question                                      answer  \
0  what is the annual fee                   The annual fee is Rs 500.   
1   how can i pay the fee  You can pay via UPI, card, or net banking.   

   confidence_score  
0                 1  
1                 1  
Single top match found with score 2:
                question                            answer  confidence_score
0  how to reset password  Go to Settings > Reset Password.                 2
